# Streaming POD vector space tutorial

In this tutorial you will learn:
- How to construct a POD vector space from snapshots that are loaded *on demand* through a `snapshot_loader`, so that the full snapshot matrix is never held in memory at once
- The roles of `block_size`, `basis_dimension`, and `oversampling`
- How to use the streaming vector space with **no scaling and no orthogonalizer** (the defaults)
- How to use the streaming vector space **with a scaler and an orthogonalizer**, and why the two are paired

## The math behind streaming POD

Let the snapshot matrix be $X \in \mathbb{R}^{N \times M}$, where each column is a state snapshot. For the tensor data used here, $N = n_{\text{var}} \, n_x$ and $M$ is the number of snapshots. Proper orthogonal decomposition (POD) seeks a rank-$k$ orthonormal basis $\Phi \in \mathbb{R}^{N \times k}$ that best approximates the snapshots,

$$
\min_{\Phi^\top \Phi = I_k} \; \lVert X - \Phi \Phi^\top X \rVert_F ,
$$

whose solution is the leading $k$ left singular vectors of $X = U \Sigma V^\top$.

For large datasets, $X$ may not fit in memory. **Streaming POD** never materializes all of $X$: it reads one column block $X_{[s:e]}$ at a time through a `snapshot_loader` callable, so the memory footprint scales with the block size, not with the total number of snapshots $M$.

`romtools` uses a randomized two-pass algorithm. Given a target rank $k$ and an oversampling parameter $p$, draw a random matrix $\Omega \in \mathbb{R}^{M \times (k+p)}$:

- **Pass 1** accumulate the range sketch one block at a time,

$$
Y = X \Omega = \sum_{\text{blocks } [s:e]} X_{[s:e]} \, \Omega_{[s:e]} ,
$$

  then orthonormalize $Y$ to obtain $Q$ (with $Q^\top Q = I$) that approximately spans the range of $X$.
- **Pass 2** form the small core matrix, again one block at a time,

$$
B = Q^\top X = \sum_{\text{blocks } [s:e]} Q^\top X_{[s:e]} ,
$$

  compute its (inexpensive) SVD $B = \tilde{U}\, \Sigma\, V^\top$, and set the POD basis $\Phi = Q \tilde{U}$, truncated to the leading $k$ columns.

Because $Q$ and $\tilde{U}$ both have orthonormal columns, $\Phi = Q\tilde{U}$ is orthonormal ($\Phi^\top \Phi = I_k$) by construction. The oversampling $p$ improves the accuracy of the randomized range sketch. The snapshots are read **twice** (one pass each); a data-derived scaler (see below) adds **one** more pass to compute its scalings.

This tutorial runs in serial. For very large problems the same construction is row-distributed across MPI ranks by passing a communicator `comm` and an SVD functor `svdFnc=vector_space.utils.SvdMethodOfSnapshots(comm)`; each rank's loader then returns only its local rows for the requested snapshot range.

In [ ]:
# First, let's import the relevant modules:
import romtools
import numpy as np
from matplotlib import pyplot as plt
from romtools import vector_space

## Loading data through a `snapshot_loader`

The streaming vector space does not take a snapshot array directly. Instead it takes a **`snapshot_loader`**: a callable `loader(start, end)` that returns the block of snapshots for the half-open range `[start, end)`. The returned array must be three-dimensional with shape `(n_var, n_dofs, end - start)` (the last axis is the snapshot axis), and the leading axes must be the same for every call.

In a real large-scale problem the loader would read each block from disk so that the full matrix is never resident in memory (see `romtools.linalg.linalg._snapshot_loader` for a file-based example). Here, for illustration, we simply slice the in-memory 1D Euler snapshots used throughout these tutorials.

In [ ]:
# Load pre-computed snapshots of the 1D Euler equations (from pressio-demo-apps).
snapshots = np.load('snapshots.npz')['snapshots']

# The snapshots are in tensor form:
n_vars, nx, nt = snapshots.shape
print('snapshot tensor shape (n_vars, nx, nt):', snapshots.shape)

# A snapshot_loader returns the 3D block for the half-open range [start, end).
# In practice this would stream each block from disk; here we slice the
# in-memory array purely for illustration.
def my_snapshot_loader(start, end):
    return snapshots[..., start:end]

## Case 1: streaming POD with no scaling and no orthogonalizer

In the simplest case we pass only the loader and the algorithm parameters. The scaler and orthogonalizer default to no-ops. We keep at most `block_size` snapshots in memory at a time, retain `basis_dimension` POD modes, and use `oversampling` extra random directions for the sketch.

Note that, unlike `VectorSpaceFromPOD`, the streaming vector space does not take a shifter: its shift (affine offset) vector is zero.

In [ ]:
block_size      = 8    # maximum number of snapshots held in memory at once
basis_dimension = 20   # number of POD modes to retain (k)
oversampling    = 5    # randomized oversampling dimension (p)

my_streaming_vector_space = vector_space.VectorSpaceFromStreamingPOD(
    snapshot_loader=my_snapshot_loader,
    block_size=block_size,
    n_snapshots=nt,
    basis_dimension=basis_dimension,
    oversampling=oversampling)

# We can view the basis and (retained) singular values:
basis = my_streaming_vector_space.get_basis()
print('The dimension of the vector space is', my_streaming_vector_space.extents())
print('Retained singular values:', my_streaming_vector_space.get_singular_values())

In [ ]:
# We can look at the density component of the first basis vector:
plt.plot(basis[0, :, 0])
plt.xlabel(r'Index')
plt.ylabel(r'$\rho$')
plt.show()

In [ ]:
# With no orthogonalizer, the streaming basis is already orthonormal in the
# Euclidean inner product (Phi^T Phi = I) by construction:
is_identity = np.einsum('ijk,ijl->kl', basis, basis)
assert np.allclose(is_identity, np.eye(basis.shape[-1]))

## Case 2: streaming POD with scaling and orthogonalization

The 1D Euler state has variables of very different magnitudes (density, momentum, and energy). A **scaler** non-dimensionalizes them before the SVD so that no single variable dominates the POD modes. Here we use a per-variable `VariableScaler("variance")`, which scales each variable by its variance across the snapshots.

Scaling changes the basis after the SVD (`post_scale`), which in general destroys the Euclidean orthonormality of the modes. We therefore pair the scaler with an **orthogonalizer** to restore $\Phi^\top \Phi = I$. `EuclideanL2Orthogonalizer` re-orthonormalizes in the standard $L^2$ inner product; if you need a physics-based inner product (e.g. weighting by cell volumes or quadrature weights $w$), use `EuclideanVectorWeightedL2Orthogonalizer(w)`, which enforces $\Phi^\top \mathrm{diag}(w)\, \Phi = I$.

Because `VariableScaler` derives its scalings from the data, it performs one additional streaming pass over the snapshots to compute them (three passes total, versus two).

In [ ]:
# Create a per-variable scaler and a Euclidean L2 orthogonalizer:
my_scaler         = vector_space.utils.VariableScaler('variance')
my_orthogonalizer = vector_space.utils.EuclideanL2Orthogonalizer()

my_scaled_vector_space = vector_space.VectorSpaceFromStreamingPOD(
    snapshot_loader=my_snapshot_loader,
    block_size=block_size,
    n_snapshots=nt,
    basis_dimension=basis_dimension,
    oversampling=oversampling,
    scaler=my_scaler,
    orthogonalizer=my_orthogonalizer)

scaled_basis = my_scaled_vector_space.get_basis()
print('The dimension of the scaled vector space is', my_scaled_vector_space.extents())

In [ ]:
# Even though scaling was applied, the orthogonalizer guarantees the basis is
# orthonormal in the Euclidean inner product:
is_identity = np.einsum('ijk,ijl->kl', scaled_basis, scaled_basis)
assert np.allclose(is_identity, np.eye(scaled_basis.shape[-1]))

In [ ]:
# Compare the density component of the first mode with and without scaling:
plt.plot(basis[0, :, 0], label='no scaling')
plt.plot(scaled_basis[0, :, 0], '--', label='variance scaling')
plt.xlabel(r'Index')
plt.ylabel(r'$\rho$')
plt.legend()
plt.show()

## Summary

- `VectorSpaceFromStreamingPOD` builds a POD basis while holding only one block of snapshots in memory at a time, loading data on demand through a `snapshot_loader`.
- `block_size` trades memory footprint against the number of loader calls; `basis_dimension` sets the retained rank $k$; `oversampling` improves the accuracy of the randomized sketch.
- With no scaler and no orthogonalizer, the basis is orthonormal by construction.
- A scaler (e.g. `VariableScaler`) should be paired with an orthogonalizer to restore orthonormality after scaling.

See the API documentation for details:
- [`VectorSpaceFromStreamingPOD`](https://pressio.github.io/rom-tools-and-workflows/romtools/vector_space.html)
- [scalers](https://pressio.github.io/rom-tools-and-workflows/romtools/vector_space/utils/scaler.html) and [orthogonalizers](https://pressio.github.io/rom-tools-and-workflows/romtools/vector_space/utils/orthogonalizer.html)